# 12 — Train Cancer-Risk CNN on BCN+MNH Merged Dataset

**Issue:** D4.5 (#141) — closed  
**Depends on:** D4.4 (#140)  
**Purpose:** Retrain the dermoscopic cancer-risk CNN on the merged BCN20000 + filtered MNH dataset using identical settings to notebook 06 (`#119`).

**Architecture / framework:** EfficientNet-B0 (ImageNet pretrained), PyTorch, via `python3 -m src.model.train`. Wraps the same training entry point used by #119 so the BCN-only vs BCN+MNH comparison in D4.6 is apples-to-apples.

**Re-runnable design (IMPORTANT):** cells 4 and 5 (smoke test + full training) skip themselves if `training_history.csv` already has ≥10 epochs. Protects the trained checkpoint from accidental overwrite when this notebook is re-executed (`nbconvert --execute`, Run-All, etc.). To force a retrain, delete `models/bcn_mnh_cancer_risk_effnet_b0/training_history.csv` first.

**Notebook numbering note:** D4.5 issue spec calls this `notebooks/09_train_bcn_mnh_cancer_risk_cnn.ipynb`, but `09`/`10`/`11` were already taken by D4.2/D4.3/D4.4 deliverables. Continuing the sequence as `12_…` to avoid renaming committed work.

## 1. Verify splits, freeze BCN test hash, log class distribution

In [11]:
import hashlib
from pathlib import Path

import pandas as pd

ROOT = Path("/Users/rehmaaziz/revela")

TRAIN_CSV     = ROOT / "splits/bcn_mnh_train.csv"
VAL_CSV       = ROOT / "splits/bcn_mnh_val.csv"
BCN_TEST_CSV  = ROOT / "data/processed/bcn20000_cancer_risk/test.csv"
CONFIG_PATH   = ROOT / "config/bcn_mnh_cancer_risk_config.yaml"
OUTPUT_DIR    = ROOT / "models/bcn_mnh_cancer_risk_effnet_b0"
HISTORY_PATH  = OUTPUT_DIR / "training_history.csv"

TRAINING_COMPLETE_MARKER_EPOCHS = 10  # full run target — used to detect prior completion

def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

bcn_test_hash_before = file_hash(BCN_TEST_CSV)

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(BCN_TEST_CSV, low_memory=False)

train_ids = set(train_df["isic_id"])
val_ids   = set(val_df["isic_id"])
test_ids  = set(test_df["isic_id"])

assert len(train_ids & test_ids) == 0, "Train contains test images — STOP"
assert len(val_ids   & test_ids) == 0, "Val contains test images — STOP"
assert len(train_ids & val_ids)  == 0, "Train and val overlap — STOP"

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,} (frozen)")
print(f"BCN test hash (frozen): {bcn_test_hash_before}")
print("All split asserts passed.\n")

print("Train class distribution:")
for cls, n in train_df["cancer_risk_class"].value_counts().items():
    print(f"  {cls:45s} {n:>6,}  ({n/len(train_df)*100:.1f}%)")

Train: 21,233 | Val: 3,619 | Test: 2,659 (frozen)
BCN test hash (frozen): a67861586e00812aadf46f2bdb4bc01b
All split asserts passed.

Train class distribution:
  Benign nevus                                  10,154  (47.8%)
  Melanoma                                       6,636  (31.3%)
  Non-melanoma skin cancer                       2,537  (11.9%)
  Other non-cancer / indeterminate lesion        1,906  (9.0%)


## 2. Preview class weights

`src/model/train.py` uses inverse-frequency class weighting: `weight_c = total / (num_classes × count_c)`. This is what notebook 06 used for BCN-only.

In [12]:
CLASS_NAMES = [
    "Melanoma",
    "Non-melanoma skin cancer",
    "Benign nevus",
    "Other non-cancer / indeterminate lesion",
]
total = len(train_df)
num_classes = len(CLASS_NAMES)

print(f"{'Class':<45} {'Count':>6}  {'%':>5}  {'Weight':>8}")
print("-" * 70)
for cls in CLASS_NAMES:
    count = (train_df["cancer_risk_class"] == cls).sum()
    pct   = count / total * 100
    w     = total / (num_classes * count) if count else 0.0
    print(f"{cls:<45} {count:>6,}  {pct:>5.1f}  {w:>8.4f}")
print("-" * 70)
print(f"{'TOTAL':<45} {total:>6,}  100.0%")

Class                                          Count      %    Weight
----------------------------------------------------------------------
Melanoma                                       6,636   31.3    0.7999
Non-melanoma skin cancer                       2,537   11.9    2.0923
Benign nevus                                  10,154   47.8    0.5228
Other non-cancer / indeterminate lesion        1,906    9.0    2.7850
----------------------------------------------------------------------
TOTAL                                         21,233  100.0%


## 3. Review training config

In [13]:
import yaml

with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

print(yaml.dump(config, default_flow_style=False, sort_keys=False).rstrip())

dataset:
  train_csv: splits/bcn_mnh_train.csv
  val_csv: splits/bcn_mnh_val.csv
  label_column: cancer_risk_class
  class_idx_column: null
  test_csv: data/processed/bcn20000_cancer_risk/test.csv
class_names:
- Melanoma
- Non-melanoma skin cancer
- Benign nevus
- Other non-cancer / indeterminate lesion
training:
  image_size: 224
  batch_size: 16
  num_workers: 0
  learning_rate: 0.0003
  weight_decay: 0.01
  epochs: 10
  use_class_weights: true
  output_dir: models/bcn_mnh_cancer_risk_effnet_b0


## 4. Smoke test (skipped if training already done)

Verifies pipeline end-to-end with 1 epoch + 2 batches. Skipped if `training_history.csv` already has ≥10 epochs so re-executing the notebook doesn't overwrite real training results.

In [14]:
import subprocess

def existing_epochs() -> int:
    if not HISTORY_PATH.exists():
        return 0
    return len(pd.read_csv(HISTORY_PATH))

if existing_epochs() >= TRAINING_COMPLETE_MARKER_EPOCHS:
    print(f"Skipping smoke test: training_history.csv already has {existing_epochs()} epochs.")
    print("To force a fresh smoke test, delete training_history.csv first.")
else:
    result = subprocess.run(
        [
            "python3", "-m", "src.model.train",
            "--config", str(CONFIG_PATH),
            "--epochs", "1",
            "--batch-size", "4",
            "--num-workers", "0",
            "--max-train-batches", "2",
            "--max-val-batches", "2",
        ],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("Smoke test failed — STOP")

Skipping smoke test: training_history.csv already has 10 epochs.
To force a fresh smoke test, delete training_history.csv first.


## 5. Full training run (skipped if training already done)

10 epochs on the full merged dataset; ~4 hours on MPS. Skipped if `training_history.csv` already has ≥10 epochs. **Do not re-run this without first deleting training_history.csv and best_model.pth** — accidental re-runs overwrite the trained checkpoint.

In [15]:
if existing_epochs() >= TRAINING_COMPLETE_MARKER_EPOCHS:
    print(f"Skipping full training: training_history.csv already has {existing_epochs()} epochs.")
    print("Existing checkpoint preserved.")
else:
    result = subprocess.run(
        ["python3", "-m", "src.model.train", "--config", str(CONFIG_PATH)],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("Full training failed — STOP")

Skipping full training: training_history.csv already has 10 epochs.
Existing checkpoint preserved.


## 6. Load training history and identify best epoch

In [16]:
history = pd.read_csv(HISTORY_PATH)
print(history.to_string(index=False, float_format="{:.4f}".format))

best_idx   = history["val_macro_f1"].idxmax()
best_epoch = int(history.loc[best_idx, "epoch"])
best_f1    = float(history.loc[best_idx, "val_macro_f1"])
best_acc   = float(history.loc[best_idx, "val_accuracy"])
best_bal   = float(history.loc[best_idx, "val_balanced_accuracy"])
print()
print(f"Best epoch:                {best_epoch}")
print(f"Best val_macro_f1:         {best_f1:.4f}")
print(f"Best val_accuracy:         {best_acc:.4f}")
print(f"Best val_balanced_accuracy:{best_bal:.4f}")

 epoch  train_loss  train_accuracy  val_loss  val_accuracy  val_macro_f1  val_balanced_accuracy
     1      0.8953          0.6406    0.7648        0.6974        0.6269                 0.6725
     2      0.7481          0.7027    0.7428        0.7168        0.6458                 0.6891
     3      0.6758          0.7330    0.7060        0.7347        0.6643                 0.6845
     4      0.6046          0.7558    0.7408        0.7162        0.6491                 0.6885
     5      0.5585          0.7708    0.7963        0.7328        0.6643                 0.6668
     6      0.5002          0.7910    0.7637        0.7416        0.6768                 0.6903
     7      0.4733          0.8003    0.7694        0.7256        0.6696                 0.6970
     8      0.4204          0.8195    0.7934        0.7394        0.6734                 0.6906
     9      0.3937          0.8313    0.9436        0.7187        0.6393                 0.6590
    10      0.3620          0.8434    0.

## 7. Mirror outputs to spec'd paths + final safety check

Issue spec wants `outputs/metrics/bcn_mnh_training_log.csv` and a checkpoint name like `models/bcn_mnh_cancer_risk_cnn_epoch{N}.pth`. `train.py` writes to `output_dir/training_history.csv` and `output_dir/best_model.pth`; we copy/rename to the spec'd names without modifying the originals.

In [17]:
import shutil

METRICS_DIR = ROOT / "outputs/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

log_dst = METRICS_DIR / "bcn_mnh_training_log.csv"
shutil.copyfile(HISTORY_PATH, log_dst)
print(f"Copied training log to {log_dst}")

ckpt_dst = ROOT / f"models/bcn_mnh_cancer_risk_cnn_epoch{best_epoch}.pth"
shutil.copyfile(OUTPUT_DIR / "best_model.pth", ckpt_dst)
print(f"Copied best checkpoint to {ckpt_dst}")

import torch
ckpt = torch.load(ckpt_dst, map_location="cpu", weights_only=False)
print(f"\nCheckpoint loads: epoch={ckpt['epoch']} val_macro_f1={ckpt['val_macro_f1']:.4f}")
print(f"class_to_idx: {ckpt['class_to_idx']}")

bcn_test_hash_after = file_hash(BCN_TEST_CSV)
assert bcn_test_hash_before == bcn_test_hash_after, \
    f"BCN test file modified during notebook execution! before={bcn_test_hash_before} after={bcn_test_hash_after}"
print(f"\nBCN test hash unchanged: {bcn_test_hash_after}")

Copied training log to /Users/rehmaaziz/revela/outputs/metrics/bcn_mnh_training_log.csv
Copied best checkpoint to /Users/rehmaaziz/revela/models/bcn_mnh_cancer_risk_cnn_epoch6.pth

Checkpoint loads: epoch=6 val_macro_f1=0.6768
class_to_idx: {'Melanoma': 0, 'Non-melanoma skin cancer': 1, 'Benign nevus': 2, 'Other non-cancer / indeterminate lesion': 3}

BCN test hash unchanged: a67861586e00812aadf46f2bdb4bc01b


## 8. Plot training curves

In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PLOT_DIR = ROOT / "outputs/plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history["epoch"], history["train_accuracy"], "o-", label="train")
axes[0].plot(history["epoch"], history["val_accuracy"],   "o-", label="val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["epoch"], history["train_loss"], "o-", label="train")
axes[1].plot(history["epoch"], history["val_loss"],   "o-", label="val")
axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history["epoch"], history["val_macro_f1"],           "o-", color="darkgreen", label="val macro-F1")
axes[2].plot(history["epoch"], history["val_balanced_accuracy"], "o-", color="darkblue",  label="val bal. acc")
axes[2].axvline(best_epoch, color="red", linestyle="--", alpha=0.5, label=f"best (epoch {best_epoch})")
axes[2].set_title("Val Macro-F1 / Balanced Acc"); axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
out_png = PLOT_DIR / "bcn_mnh_training_curves.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved: {out_png}")

Saved: /Users/rehmaaziz/revela/outputs/plots/bcn_mnh_training_curves.png


## 9. Final summary

In [19]:
print("=== D4.5 final summary ===")
print(f"  Train images:           {len(train_df):,}")
print(f"  Val images:             {len(val_df):,}")
print(f"  Test (frozen):          {len(test_df):,}")
print(f"  Epochs run:             {len(history)}")
print(f"  Best epoch:             {best_epoch}")
print(f"  Best val_macro_f1:      {best_f1:.4f}")
print(f"  Best val_accuracy:      {best_acc:.4f}")
print(f"  Best val_balanced_acc:  {best_bal:.4f}")
print(f"  Checkpoint:             models/bcn_mnh_cancer_risk_cnn_epoch{best_epoch}.pth")
print(f"  Native checkpoint:      models/bcn_mnh_cancer_risk_effnet_b0/best_model.pth")
print(f"  Training log:           outputs/metrics/bcn_mnh_training_log.csv")
print(f"  BCN test hash:          {bcn_test_hash_after} (unchanged)")
print(f"\nComparison vs BCN-only #119:")
print(f"  Val macro-F1:     BCN-only 0.6924 → BCN+MNH {best_f1:.4f}  ({best_f1-0.6924:+.4f})")
print(f"  Val accuracy:     BCN-only 0.7009 → BCN+MNH {best_acc:.4f}  ({best_acc-0.7009:+.4f})")
print(f"  Val balanced acc: BCN-only 0.6942 → BCN+MNH {best_bal:.4f}  ({best_bal-0.6942:+.4f})")

=== D4.5 final summary ===
  Train images:           21,233
  Val images:             3,619
  Test (frozen):          2,659
  Epochs run:             10
  Best epoch:             6
  Best val_macro_f1:      0.6768
  Best val_accuracy:      0.7416
  Best val_balanced_acc:  0.6903
  Checkpoint:             models/bcn_mnh_cancer_risk_cnn_epoch6.pth
  Native checkpoint:      models/bcn_mnh_cancer_risk_effnet_b0/best_model.pth
  Training log:           outputs/metrics/bcn_mnh_training_log.csv
  BCN test hash:          a67861586e00812aadf46f2bdb4bc01b (unchanged)

Comparison vs BCN-only #119:
  Val macro-F1:     BCN-only 0.6924 → BCN+MNH 0.6768  (-0.0156)
  Val accuracy:     BCN-only 0.7009 → BCN+MNH 0.7416  (+0.0407)
  Val balanced acc: BCN-only 0.6942 → BCN+MNH 0.6903  (-0.0039)


## D5.1 — Inference Registration Status

**Issue:** #163 — D5.1 Register improved dermoscopic model for local inference
**Completed:** 2026-05-20
**Model ID:** `dermoscopic_cancer_risk_bcn_mnh_v1`

The BCN+MNH checkpoint produced by this notebook has been registered in `src/inference/model_registry.py` and is available for local inference via the existing adapter.

| Item | Value |
|---|---|
| Registry key | `dermoscopic_cancer_risk_bcn_mnh_v1` |
| Checkpoint | `models/bcn_mnh_cancer_risk_effnet_b0/best_model.pth` |
| Architecture | EfficientNet-B0 / 224×224 |
| Smoke test | PASSED — top prediction: Benign nevus (51.89%), all 10 schema fields present |
| Smoke test script | `scripts/smoke_test_dermoscopic_inference.py` |

**Next step:** Streamlit UI enablement is tracked separately — this model is not yet wired into the app UI (out of scope for D5.1).